# Combined Var1+Var5 with MSE-only Loss

Combines Var1's training recipe (BS=400, linear warmup + cosine anneal LR) with Var5's deeper / dropout-regularised RefineNet decoder. The mixed MSE+NMSE loss is dropped — pure MSE is used throughout so absolute reconstruction magnitude is anchored.

This notebook is a standalone, simplified version of `var_combined_v1v5_mse.py` and follows the same flow as `4 feb/ADJSCC-CSInet+.ipynb`:
1. dataset
2. AF module
3. ATN module
4. encoder
5. real → complex symbols + power normalisation
6. wireless channel
7. complex → real (C2R)
8. decoder
9. STN
10. training loop


## Imports and seed

In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Config

In [ ]:
class TrainingConfig:
    train_file:         str   = "train_data.mat"
    val_file:           str   = "val_data.mat"
    test_file:          str   = "test_data.mat"
    checkpoint_dir:     str   = "checkpoints_combined_v1v5_mse"

    fast_dev_run:       bool  = False
    run_training:       bool  = False
    run_evaluation:     bool  = False
    checkpoint_path:    str | None = None
    resume_latest:      bool  = False

    # ── Training schedule (from Var1) ─────────────────────────────────────────
    epochs:             int   = 500
    batch_size:         int   = 400          # Var1: was 200
    learning_rate:      float = 0.001        # peak LR reached after warmup
    min_lr:             float = 0.0001       # cosine decays down to this
    warmup_epochs:      int   = 30           # Var1: was 20
    save_every:         int   = 10

    # ── Regularisation (from Var5) ────────────────────────────────────────────
    weight_decay:       float = 5e-5         # Var5: was 1e-5
    grad_clip:          float = 1.0

    # ── Architecture ──────────────────────────────────────────────────────────
    k_feedback:         int   = 64
    compression_ratio:  int   = 16
    snr_low:            float = -10.0
    snr_high:           float = 10.0
    decoder_blocks:     int   = 8            # Var5: was 5
    decoder_dropout_p:  float = 0.10         # Var5: was 0 (no dropout)

    # ── Loss — MSE ONLY throughout all epochs ─────────────────────────────────
    # (Var1 and Var5 both used mixed MSE+NMSE after warmup_epochs)
    # Setting nmse_weight=0 and warmup_epochs=epochs forces pure MSE always.
    mse_weight:         float = 1.0
    nmse_weight:        float = 0.0          # disabled
    warmup_epochs_loss: int   = 500          # effectively never switches
cfg = TrainingConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)


## Dataset

Streams the QuaDRiGa CSI HDF5 files lazily and applies a global per-channel scale computed on the train split.

In [ ]:
def _read_scalar(dataset):
    value = dataset[()]
    if isinstance(value, np.ndarray) and value.size == 1:
        return float(value.reshape(-1)[0])
    return value

def load_dataset_cfg(path: str):
    with h5py.File(path, "r") as f:
        group = f["cfg"]
        return {k: _read_scalar(group[k]) for k in group.keys()
                if isinstance(group[k], h5py.Dataset)}

def get_train_global_scale(train_path: str, chunk_size: int = 500):
    stats = {"dl": {"sum_sq": 0.0, "count": 0},
             "ul": {"sum_sq": 0.0, "count": 0}}
    with h5py.File(train_path, "r") as f:
        for key, sn in [("csi_dl", "dl"), ("csi_ul", "ul")]:
            ds = f[key]; N = ds.shape[3]
            for start in range(0, N, chunk_size):
                chunk = ds[:, :, :, start:min(start + chunk_size, N)]
                r  = chunk["real"].astype(np.float32)
                im = chunk["imag"].astype(np.float32)
                stats[sn]["sum_sq"] += float(np.sum(r**2) + np.sum(im**2))
                stats[sn]["count"]  += r.size + im.size
    out = {}
    for k in ("dl", "ul"):
        var = stats[k]["sum_sq"] / max(stats[k]["count"], 1)
        out[k] = {"std": float(np.sqrt(var + 1e-12))}
    print("Scale stats:", out)
    return out

class CSIDatasetManager:
    def __init__(self, train_path, val_path, test_path, stats):
        self.stats = stats
        self.paths = {"train": train_path, "val": val_path, "test": test_path}
        self.files = {}; self.datasets = {}; self.lengths = {}
        for split, path in self.paths.items():
            h = h5py.File(path, "r")
            self.files[split]    = h
            self.datasets[split] = {"dl": h["csi_dl"], "ul": h["csi_ul"]}
            self.lengths[split]  = int(h["csi_dl"].shape[3])
            print(f"{split}: {self.lengths[split]} samples")

    def close(self):
        for h in self.files.values(): h.close()

    def _normalize(self, arr, k): return arr / (self.stats[k]["std"] + 1e-8)
    def denormalize(self, tensor, k): return tensor * (self.stats[k]["std"] + 1e-8)

    def _process(self, arr, k, normalize=True):
        r  = arr["real"].astype(np.float32)
        im = arr["imag"].astype(np.float32)
        if normalize:
            r  = self._normalize(r, k)
            im = self._normalize(im, k)
        merged = np.stack([r, im], axis=2)
        merged = np.squeeze(merged, axis=3)
        return np.transpose(merged, (3, 2, 0, 1))

    def get_batch(self, split, indices, snr_values=None):
        indices = np.sort(np.asarray(indices, dtype=np.int64))
        dl  = torch.from_numpy(self._process(
              self.datasets[split]["dl"][:, :, :, indices], "dl")).float()
        ul  = torch.from_numpy(self._process(
              self.datasets[split]["ul"][:, :, :, indices], "ul")).float()
        if snr_values is None:
            snr_values = np.random.uniform(
                cfg.snr_low, cfg.snr_high, (len(indices), 1)).astype(np.float32)
        else:
            snr_values = np.asarray(snr_values, dtype=np.float32).reshape(len(indices), 1)
        return dl, ul, torch.from_numpy(snr_values).float()

    def iterate_split(self, split, batch_size, shuffle=False,
                      generator=None, fixed_snr=None):
        total = self.lengths[split]
        order = np.arange(total, dtype=np.int64)
        if shuffle:
            rng = generator if generator is not None else np.random.default_rng()
            rng.shuffle(order)
        for start in range(0, total, batch_size):
            bi = order[start:start + batch_size]
            sv = (None if fixed_snr is None
                  else np.full((len(bi), 1), fixed_snr, dtype=np.float32))
            yield self.get_batch(split, bi, snr_values=sv)

In [ ]:
# Set these paths to the QuaDRiGa CSI .mat files on your machine.
train_file = "train_data.mat"
val_file   = "val_data.mat"
test_file  = "test_data.mat"


In [ ]:
stats = get_train_global_scale(train_file)
dataset = CSIDatasetManager(train_file, val_file, test_file, stats)


## AF Module

Channel-wise SNR-aware feature recalibration: GAP over (H,W), concat with SNR (dB), 2-layer MLP → sigmoid → per-channel scale.

In [ ]:
class AFModule(nn.Module):
    """Attention Feature module: rescales channels conditioned on SNR."""
    def __init__(self, channels, reduction_ratio=2):
        super().__init__()
        hidden = max(channels // reduction_ratio, 1)
        self.fc1 = nn.Linear(channels + 1, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x, snr):
        pooled = F.adaptive_avg_pool2d(x, 1).flatten(1)
        scale  = torch.sigmoid(
            self.fc2(F.relu(self.fc1(torch.cat([pooled, snr], dim=1))))
        ).view(x.size(0), x.size(1), 1, 1)
        return x * scale

## ATN — Analysis Transform Network

Three-layer (or wider, in deeper variants) conv stack with asymmetric strides that compresses the 32×32 angular-delay map to the truncated representation used by the SC-CSI encoder.

In [ ]:
class ATN(nn.Module):
    """Analysis Transform Network — spatial-frequency → transform domain."""
    def __init__(self):
        super().__init__()
        self.conv1  = nn.Conv2d(2, 16, 3, stride=(2, 1), padding=1)
        self.bn1    = nn.BatchNorm2d(16); self.prelu1 = nn.PReLU(); self.af1 = AFModule(16)
        self.conv2  = nn.Conv2d(16, 16, 3, stride=(2, 1), padding=1)
        self.bn2    = nn.BatchNorm2d(16); self.prelu2 = nn.PReLU(); self.af2 = AFModule(16)
        self.conv3  = nn.Conv2d(16, 2,  3, stride=(2, 1), padding=1)
        self.bn3    = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.conv1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.conv2(x))), snr)
        return self.bn3(self.conv3(x))

## Encoder — CSINet+ encoder with AF modules

Two 7×7 conv blocks with AF modules, then a fully-connected layer projects the flattened map to the M-dimensional real-valued codeword.

In [ ]:
class CsiNetPlusEncoderWithAF(nn.Module):
    """CsiNet+ style encoder with AF modules."""
    def __init__(self, compression_ratio):
        super().__init__()
        self.total_elements = 2 * 32 * 32
        self.M              = self.total_elements // compression_ratio
        self.conv1 = nn.Conv2d(2, 2, 7, padding=3); self.bn1 = nn.BatchNorm2d(2); self.af1 = AFModule(2)
        self.conv2 = nn.Conv2d(2, 2, 7, padding=3); self.bn2 = nn.BatchNorm2d(2); self.af2 = AFModule(2)
        self.fc    = nn.Linear(self.total_elements, self.M)

    def forward(self, x, snr):
        x = self.af1(F.leaky_relu(self.bn1(self.conv1(x)), 0.3), snr)
        x = self.af2(F.leaky_relu(self.bn2(self.conv2(x)), 0.3), snr)
        return self.fc(x.flatten(1))

## Real → complex symbols + power normalisation

Splits the M real outputs into an M/2-length complex vector and rescales it to unit average power per symbol.

In [ ]:
def enc_to_complex_and_normalize(encoder_output):
    k    = encoder_output.shape[1] // 2
    s    = torch.complex(encoder_output[:, :k], encoder_output[:, k:])
    pwr  = torch.mean(s.abs().square(), dim=1, keepdim=True)
    return s / torch.sqrt(pwr + 1e-8)

## Wireless channel

Differentiable OFDM AWGN channel: picks `k` uplink subcarriers, transmits the complex symbols, adds Gaussian noise scaled to the requested SNR, and applies maximum-ratio combining at the BS.

In [ ]:
class WirelessChannelSimulator(nn.Module):
    def __init__(self, num_bs_antennas=32, training_random_subcarriers=True):
        super().__init__()
        self.Nt  = num_bs_antennas
        self.trs = training_random_subcarriers

    def _select_indices(self, num_sub, k, dev):
        if self.training and self.trs:
            return torch.randperm(num_sub, device=dev)[:k]
        return torch.linspace(0, num_sub - 1, k, device=dev).round().long()

    def forward(self, s, snr_db, h_uplink):
        bs, k = s.shape; dev = s.device
        idx     = self._select_indices(h_uplink.shape[2], k, dev)
        h_s     = h_uplink[:, :, idx, :]
        h_u     = torch.complex(h_s[:, 0], h_s[:, 1])
        snr_lin = torch.pow(10.0, snr_db / 10.0)
        nstd    = torch.sqrt(1.0 / snr_lin / 2.0).unsqueeze(-1)
        z = torch.complex(
            torch.randn(bs, k, self.Nt, device=dev) * nstd,
            torch.randn(bs, k, self.Nt, device=dev) * nstd,
        )
        y = h_u * s.unsqueeze(-1) + z
        w = h_u / (torch.norm(h_u, dim=2, keepdim=True) + 1e-8)
        return torch.sum(torch.conj(w) * y, dim=2)

## C2R — Complex → real for the decoder

In [ ]:
class ComplexToReal(nn.Module):
    def forward(self, s): return torch.cat([s.real, s.imag], dim=1)

## Decoder — CSINet+ RefineNet stack

FC → 32×32 feature map, an initial conv block, then a chain of RefineNet residual blocks (each conv block is followed by an AF module).

In [ ]:
class ModifiedRefineNetBlock(nn.Module):
    """
    Residual RefineNet block with Dropout2d after the first activation.
    Spatial dropout zeroes entire feature-map channels — more effective
    than per-element dropout for convolutional representations.
    Dropout is applied only during training (standard nn.Dropout2d behaviour).
    """
    def __init__(self, channels, dropout_p: float = 0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, 8,       7, padding=3)
        self.bn1   = nn.BatchNorm2d(8);   self.af1 = AFModule(8)
        self.drop  = nn.Dropout2d(p=dropout_p)   # ← Var5 addition

        self.conv2 = nn.Conv2d(8,       16,       5, padding=2)
        self.bn2   = nn.BatchNorm2d(16); self.af2 = AFModule(16)

        self.conv3 = nn.Conv2d(16, channels,      3, padding=1)
        self.bn3   = nn.BatchNorm2d(channels);    self.af3 = AFModule(channels)

    def forward(self, x, snr):
        residual = x
        x = self.drop(self.af1(F.leaky_relu(self.bn1(self.conv1(x)), 0.3), snr))
        x = self.af2(F.leaky_relu(self.bn2(self.conv2(x)), 0.3), snr)
        x = self.af3(self.bn3(self.conv3(x)), snr)
        return residual + x

In [ ]:
class CsiNetPlusDecoder(nn.Module):
    """
    CsiNet+ style decoder.
    num_blocks=8 (Var5), dropout_p=0.10 (Var5).
    """
    def __init__(self, input_dim, height=32, width=32, channels=2,
                 num_blocks: int = 8, dropout_p: float = 0.10):
        super().__init__()
        self.height       = height
        self.width        = width
        self.channels     = channels
        self.flattened_dim = height * width * channels

        self.fc           = nn.Linear(input_dim, self.flattened_dim)
        self.initial_conv = nn.Conv2d(channels, channels, 7, padding=3)
        self.initial_bn   = nn.BatchNorm2d(channels)
        self.initial_af   = AFModule(channels)
        self.refinenet_chain = nn.ModuleList([
            ModifiedRefineNetBlock(channels, dropout_p=dropout_p)
            for _ in range(num_blocks)
        ])

    def forward(self, x, snr):
        x = self.fc(x).view(-1, self.channels, self.height, self.width)
        x = self.initial_af(
            F.leaky_relu(self.initial_bn(self.initial_conv(x)), 0.3), snr
        )
        for block in self.refinenet_chain:
            x = block(x, snr)
        return x

## STN — Synthesis Transform Network

Mirror image of the ATN: transposed-conv stack that expands the latent back to the 32×32 angular-delay map.

In [ ]:
class STN(nn.Module):
    """Synthesis Transform Network — transform domain → spatial-frequency."""
    def __init__(self):
        super().__init__()
        self.tc1   = nn.ConvTranspose2d(2, 16, 3, stride=(2,1), padding=1, output_padding=(1,0))
        self.bn1   = nn.BatchNorm2d(16); self.prelu1 = nn.PReLU(); self.af1 = AFModule(16)
        self.tc2   = nn.ConvTranspose2d(16, 16, 3, stride=(2,1), padding=1, output_padding=(1,0))
        self.bn2   = nn.BatchNorm2d(16); self.prelu2 = nn.PReLU(); self.af2 = AFModule(16)
        self.tc3   = nn.ConvTranspose2d(16, 2,  3, stride=(2,1), padding=1, output_padding=(1,0))
        self.bn3   = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.tc1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.tc2(x))), snr)
        return self.bn3(self.tc3(x))

## Build the modules

In [ ]:
dataset_cfg = load_dataset_cfg(train_file)
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=cfg.compression_ratio).to(device)
channel_sim = WirelessChannelSimulator(num_bs_antennas=int(dataset_cfg['num_bs_antennas'])).to(device)
c2r = ComplexToReal().to(device)
decoder = CsiNetPlusDecoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = (list(atn.parameters()) + list(encoder.parameters())
              + list(decoder.parameters()) + list(stn.parameters()))
print('Total trainable parameters:', sum(p.numel() for p in all_params if p.requires_grad))


## Training loop

In [ ]:
def samplewise_linear_nmse(h_true, h_pred):
    num = torch.sum((h_true - h_pred)**2, dim=(1, 2, 3))
    den = torch.sum(h_true**2,            dim=(1, 2, 3)) + 1e-8
    return num / den

def nmse_db_from_sums(error_sum, power_sum):
    return 10.0 * math.log10((error_sum / max(power_sum, 1e-12)) + 1e-12)

def forward_pass(H_d, H_u, snr):
    T     = atn(H_d, snr)
    c     = encoder(T, snr)
    s     = enc_to_complex_and_normalize(c)
    s_hat = channel_sim(s, snr, H_u)
    c_hat = c2r(s_hat)
    T_hat = decoder(c_hat, snr)
    return stn(T_hat, snr)

def compute_loss(H_true, H_pred):
    """
    MSE-only loss — no NMSE term, no epoch-gating.
    This is the key difference from Var1 and Var5 which both switch
    to a mixed MSE+NMSE objective after warmup_epochs.
    """
    mse_loss    = mse_criterion(H_pred, H_true)
    # Still compute NMSE for logging purposes (not part of backward graph)
    with torch.no_grad():
        nmse_linear = samplewise_linear_nmse(H_true, H_pred).mean()
    return mse_loss, mse_loss.detach(), nmse_linear.detach()

def run_epoch(split: str, epoch_index: int = 0, fixed_snr=None):
    is_train = split == "train"
    for m in [atn, encoder, decoder, stn, channel_sim]:
        m.train(is_train)

    rng         = np.random.default_rng(SEED + epoch_index)
    total_loss  = total_mse = total_samples = 0.0
    error_sum   = power_sum = 0.0

    for batch_idx, (H_d, H_u, snr) in enumerate(
        dataset.iterate_split(
            split, cfg.batch_size,
            shuffle=is_train,
            generator=rng if is_train else None,
            fixed_snr=fixed_snr,
        )
    ):
        if is_train and debug_train_batches is not None and batch_idx >= debug_train_batches:
            break
        if not is_train and debug_eval_batches is not None and batch_idx >= debug_eval_batches:
            break

        H_d = H_d.to(device, non_blocking=True)
        H_u = H_u.to(device, non_blocking=True)
        snr = snr.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            H_hat = forward_pass(H_d, H_u, snr)
            total_batch_loss, mse_loss, nmse_linear = compute_loss(H_d, H_hat)
            if is_train:
                total_batch_loss.backward()
                if cfg.grad_clip > 0:
                    nn.utils.clip_grad_norm_(all_params, cfg.grad_clip)
                optimizer.step()

        bs           = H_d.size(0)
        total_loss   += float(total_batch_loss.detach()) * bs
        total_mse    += float(mse_loss)                  * bs
        total_samples += bs

        H_d_dn  = dataset.denormalize(H_d.detach(),   "dl")
        H_hat_dn = dataset.denormalize(H_hat.detach(), "dl")
        error_sum += float(torch.sum((H_d_dn - H_hat_dn)**2))
        power_sum += float(torch.sum(H_d_dn**2))

    return {
        "loss":        total_loss  / max(total_samples, 1),
        "mse":         total_mse   / max(total_samples, 1),
        "nmse_db":     nmse_db_from_sums(error_sum, power_sum),
        "linear_nmse": error_sum   / max(power_sum, 1e-12),
        "samples":     int(total_samples),
    }

def save_checkpoint(epoch: int, best_linear_nmse: float, tag: str):
    torch.save(
        {
            "epoch":                epoch,
            "cfg":                  cfg.__dict__,
            "stats":                stats,
            "best_linear_nmse":     best_linear_nmse,
            "atn_state_dict":       atn.state_dict(),
            "encoder_state_dict":   encoder.state_dict(),
            "decoder_state_dict":   decoder.state_dict(),
            "stn_state_dict":       stn.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        },
        Path(cfg.checkpoint_dir) / tag,
    )
    print(f"  Saved: {Path(cfg.checkpoint_dir) / tag}")

def find_latest_checkpoint():
    d      = Path(cfg.checkpoint_dir)
    cands  = list(d.glob("epoch_*.pth")) + [d / "final_model.pth", d / "best_model.pth"]
    cands  = [p for p in cands if p.exists()]
    if not cands:
        raise FileNotFoundError(f"No checkpoints found in {d}")
    return max(cands, key=lambda p: p.stat().st_mtime)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    atn.load_state_dict(ckpt["atn_state_dict"])
    encoder.load_state_dict(ckpt["encoder_state_dict"])
    decoder.load_state_dict(ckpt["decoder_state_dict"])
    stn.load_state_dict(ckpt["stn_state_dict"])
    if "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    print(f"  Loaded: {path}")
    return ckpt

def evaluate_snr_sweep(snr_points, split="test"):
    results = []
    for snr_db in snr_points:
        m = run_epoch(split, fixed_snr=snr_db)
        results.append(m["nmse_db"])
        print(f"  SNR {snr_db:>4} dB → NMSE {m['nmse_db']:.3f} dB")
    return results

### Optimiser, scheduler and loss weights

These cells reproduce the variant's exact training recipe — open the source `.py` for the line-by-line argparse / CLI logic.

In [ ]:
# optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
#                                                   factor=0.5, patience=cfg.patience,
#                                                   min_lr=cfg.min_lr)
# mse_criterion = nn.MSELoss()
# (See the .py for any variant-specific overrides — e.g. cosine LR
#  schedules, AdamW, or per-parameter-group weight decay.)


### Run training

```python
for epoch in range(cfg.epochs):
    train_metrics = run_epoch('train', epoch_index=epoch)
    val_metrics   = run_epoch('val',   epoch_index=epoch)
    # scheduler.step(val_metrics['linear_nmse'])
```

After training, sweep test NMSE over a fixed SNR grid:

```python
snr_points = [-10, -5, 0, 5, 10]
nmse_db = evaluate_snr_sweep(snr_points, split='test')
```